In [1]:
import pandas as pd
import numpy as np
import json

## Define the metadata

In [4]:
import json
with open("../citation.json", 'r') as f:
    citation = json.load(f)

In [5]:
citation

{'doi': 'https://doi.org/10.1038/s42255-019-0152-6',
 'citation': 'Vijay, J., Gauthier, MF., Biswell, R.L. et al. Single-cell analysis of human adipose tissue identifies depot- and disease-specific cell types. Nat Metab 2, 97–109 (2020).',
 'tables': ['https://static-content.springer.com/esm/art%3A10.1038%2Fs42255-019-0152-6/MediaObjects/42255_2019_152_MOESM3_ESM.xlsx'],
 'organ': 'adipose',
 'species': 'homo_sapiens',
 'reference': 'GRCh38'}

In [27]:
organism = "homo_sapiens"
cell_source = "adipose"
cell_state = None

sheet_name = "Supplementary Table 2"
table_name = "clean_degs.xlsx" # local name
ct_map_sheeet_name = "Supplementary Table 3"

## Read in file

In [13]:
col_rename = {
    "P value": "p_raw",
    "Average LogFC": "log_fc",
    "Percentage of cells in cluster expressing the gene": "nnz_frac",
    "Percentage of cells in all other clusters expressing the gene": "nnz_frac_out",
    "Adj. P value": "p_corr",
    "Cluster#": "celltype_num",
    "Gene": "target",
    "Cluster name": "group"
}

col_rename_ct = {
    "Cluster": "group",
    "Manual Annotation": "group_name",
    "Unsupervised Annotation1 (a)": "group_name_detailed"
}

In [28]:
name_map = pd.read_excel(table_name, sheet_name=ct_map_sheeet_name, skiprows=2)#.rename(columns=col_rename_ct).dropna()


In [35]:
expanded_df = (
    name_map.assign(cluster=name_map['Clusters'].str.split(','))  # Split the comma-separated strings
    .explode('cluster')  # Expand the lists into individual rows
    .reset_index(drop=True)  # Reset the index
)

In [42]:
expanded_df.drop_duplicates(subset=["Annotation"]).drop(columns=["Clusters", "Reference", "Gene"]).set_index("cluster")

,Annotation
cluster,
E1,Endothelial cells
E1,Microvascular endothelial cells of adipose tissue
E3,Lymphatic endothelial cells
IS1,Naive T cells
IS6,Activated T cells
IS2,Macrophage
IS2,Adipose tissue macrophages involved in lipid m...
IS9,Macrophage polarization
IS10,CD16- monocytes


In [47]:
np.sort(df.group.unique())

array(['E1', 'E2', 'E3', 'I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'P1',
       'P2', 'P3', 'P4', 'P5', 'P6', 'P7'], dtype=object)

In [24]:
df = pd.read_excel(table_name, sheet_name=sheet_name, skiprows=2).rename(columns=col_rename)
name_map = pd.read_excel(table_name, sheet_name=ct_map_sheeet_name, skiprows=2).rename(columns=col_rename_ct).dropna()
name_map = name_map[["group", "group_name", "group_name_detailed"]].drop_duplicates().set_index("group")
df['group'] = df.group.map(name_map["group_name_detailed"])

InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [15]:
df.head()

,p_raw,log_fc,nnz_frac,nnz_frac_out,p_corr,celltype_num,target,group
0,0.0,2.365577,0.824,0.088,0.0,0,KRT18,Progenitor/ Stem cells
1,0.0,2.050661,0.674,0.059,0.0,0,KRT8,Progenitor/ Stem cells
2,0.0,1.911774,0.833,0.111,0.0,0,ITLN1,Progenitor/ Stem cells
3,0.0,1.718079,0.774,0.123,0.0,0,TM4SF1,Progenitor/ Stem cells
4,0.0,1.713866,0.542,0.052,0.0,0,KRT19,Progenitor/ Stem cells


## Perform the filtering

In [16]:
col_pval = "p_corr"
col_lfc = "log_fc"
col_gene = "target"
col_ct = "group"
col_rank = "nnz_frac"

In [19]:
min_mean = 100
max_pval = 1e-10
min_lfc = 2.2
max_gene_shares = 2
max_per_celltype = 20

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)].sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [20]:
markers[col_ct].value_counts()

Immune cells              20
Endothelial cells          9
Progenitor/ Stem cells     8
Name: group, dtype: int64

In [21]:
markers.groupby(col_ct)[col_rank].mean().sort_values()

group
Endothelial cells         0.594667
Progenitor/ Stem cells    0.655250
Immune cells              0.801150
Name: nnz_frac, dtype: float64

## Convert to evidence json

In [22]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "gene"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
save = df.to_dict(orient = "records")

## Save evidence

In [73]:
with open("evidence.json", "w") as f:
    json.dump(save, f, indent = 4) 